In [12]:
pip install --upgrade opencv-python mediapipe numpy yt-dlp

  Using cached mediapipe-0.10.21-cp311-cp311-win_amd64.whl.metadata (10 kB)
  Using cached numpy-2.3.5-cp311-cp311-win_amd64.whl.metadata (60 kB)
INFO: pip is looking at multiple versions of mediapipe to determine which version is compatible with other requirements. This could take a while.
  Using cached mediapipe-0.10.20-cp311-cp311-win_amd64.whl.metadata (9.9 kB)
  Using cached mediapipe-0.10.18-cp311-cp311-win_amd64.whl.metadata (9.9 kB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
import pathlib
import logging
import yt_dlp

logging.basicConfig(level=logging.INFO)

def download_clip(
    url: str,
    output_dir: pathlib.Path,
    start_sec: int | None = None,
    end_sec: int | None = None,
    basename: str | None = None,
) -> pathlib.Path | None:
    """
    Download a clip (full or time-sliced) to output_dir.
    Returns the local mp4 path or None on failure.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    # Default basename from video id if not provided
    if basename is None:
        basename = "clip"

    # Explicit template; yt-dlp will append extension
    outtmpl = str(output_dir / f"{basename}.%(ext)s")

    # Build options
    ydl_opts = {
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4/best",
        "outtmpl": outtmpl,
        "merge_output_format": "mp4",
        "quiet": True,
        "noprogress": True,
    }
    if start_sec is not None and end_sec is not None:
        # Sections format requires a stream selector; "*" means all streams
        ydl_opts["download_sections"] = [f"*{start_sec}-{end_sec}"]

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            # Returns int (number of downloads), not paths
            ret = ydl.download([url])
            if ret != 0:
                logging.error(f"yt-dlp returned non-zero status: {ret}")
                return None
    except yt_dlp.DownloadError as e:
        logging.error(f"Download error for {url}: {e}")
        return None

    # Resolve actual saved file: outtmpl results in mp4
    saved = output_dir / f"{basename}.mp4"
    if not saved.exists():
        logging.warning(f"Expected output missing: {saved}")
        # Some cases produce different names; try to find any mp4
        candidates = list(output_dir.glob(f"{basename}*.mp4"))
        if candidates:
            return candidates[0].resolve()
        return None

    return saved.resolve()

In [14]:
!yt-dlp "https://www.youtube.com/watch?v=Ot-rBhiIUKs" -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4" --download-sections "*80-120" -o instructor.mp4

[youtube] Extracting URL: https://www.youtube.com/watch?v=Ot-rBhiIUKs
[youtube] Ot-rBhiIUKs: Downloading webpage
[youtube] Ot-rBhiIUKs: Downloading android sdkless player API JSON
[youtube] Ot-rBhiIUKs: Downloading web safari player API JSON
[youtube] Ot-rBhiIUKs: Downloading m3u8 information
[info] Ot-rBhiIUKs: Downloading 1 format(s): 399+140
[info] Ot-rBhiIUKs: Downloading 1 time ranges: 80.0-120.0
[download] instructor.mp4 has already been downloaded


In [27]:
def download_clip(url: str,
                  output_dir: pathlib.Path,
                  start_sec: int | None = None,
                  end_sec: int | None = None,
                  basename: str = "clip") -> pathlib.Path | None:
    output_dir.mkdir(parents=True, exist_ok=True)
    outtmpl = str(output_dir / f"{basename}.%(ext)s")

    saved = output_dir / f"{basename}.mp4"
    if saved.exists():
        saved.unlink()  # delete old file

    ydl_opts = {
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4",
        "outtmpl": outtmpl,
        "merge_output_format": "mp4",
        "quiet": True,
        "noprogress": True,
        "overwrites": True,
    }

    # 🔥 Correct way to set download_sections
    if start_sec is not None and end_sec is not None:
        ydl_opts["download_sections"] = [f"*{start_sec}-{end_sec}"]

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ret = ydl.download([url])
        if ret != 0:
            return None

    return saved if saved.exists() else None

In [21]:
import cv2
import numpy as np
import mediapipe as mp
from typing import List, Tuple

def load_video_frames(path: pathlib.Path, max_frames: int | None = None) -> List[np.ndarray]:
    """Read frames (BGR) from video path. Optionally cap frame count."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {path}")
    frames = []
    count = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
        count += 1
        if max_frames is not None and count >= max_frames:
            break
    cap.release()
    return frames

def extract_pose_landmarks(frames: List[np.ndarray]) -> List[np.ndarray]:
    """
    Returns a list of (N_landmarks x 2) arrays with normalized (x,y) in [0,1].
    Skips frames with no detection.
    """
    mp_pose = mp.solutions.pose
    results_list: List[np.ndarray] = []
    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        enable_segmentation=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as pose:
        for frame_bgr in frames:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            result = pose.process(frame_rgb)
            if result.pose_landmarks is None:
                continue
            lm = result.pose_landmarks.landmark
            arr = np.array([[p.x, p.y] for p in lm], dtype=np.float32)  # shape: (33,2)
            results_list.append(arr)
    return results_list

def draw_skeleton(frame_bgr: np.ndarray, landmarks_xy01: np.ndarray) -> np.ndarray:
    """
    Draw simple skeleton (points + selected lines) on a BGR frame.
    landmarks_xy01: (N,2) normalized coordinates in [0,1].
    """
    h, w = frame_bgr.shape[:2]
    pts = (landmarks_xy01 * np.array([w, h], dtype=np.float32)).astype(int)

    # Draw points
    out = frame_bgr.copy()
    for (x, y) in pts:
        cv2.circle(out, (x, y), 2, (0, 255, 0), -1)

    # Minimal connections (shoulders, elbows, hips, knees, ankles)
    # MediaPipe indices: https://developers.google.com/mediapipe/solutions/vision/pose_landmarker#pose-landmarks
    C = [
        (11, 12),  # shoulders
        (11, 13), (13, 15),  # left arm
        (12, 14), (14, 16),  # right arm
        (23, 24),  # hips
        (23, 25), (25, 27),  # left leg
        (24, 26), (26, 28),  # right leg
        (27, 29), (28, 30),  # shins -> ankles
    ]
    for i, j in C:
        if i < len(pts) and j < len(pts):
            cv2.line(out, tuple(pts[i]), tuple(pts[j]), (255, 0, 0), 2)

    return out


In [22]:
from dataclasses import dataclass

# Use meaningful joints: shoulders, elbows, wrists, hips, knees, ankles
STABLE_IDX = np.array([11,12,13,14,15,16,23,24,25,26,27,28,29,30], dtype=int)

@dataclass
class SimilarityResult:
    score: float
    per_joint_dist: np.ndarray  # shape (M,)

def align_skeletons(A: np.ndarray, B: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Align A and B by subtracting centroid and scaling by RMS distance
    to remove translation/scale effects. Returns (A_aligned, B_aligned).
    A, B: (N,2) normalized coords.
    """
    def center_scale(X: np.ndarray) -> np.ndarray:
        C = X.mean(axis=0, keepdims=True)
        X0 = X - C
        s = np.sqrt((X0**2).sum() / X0.shape[0]) + 1e-8
        return X0 / s
    return center_scale(A), center_scale(B)

def similarity_score(A: np.ndarray, B: np.ndarray) -> SimilarityResult:
    """
    Compute similarity after alignment on stable joints.
    Returns score in [0,100], higher is more similar.
    """
    A_sel = A[STABLE_IDX]
    B_sel = B[STABLE_IDX]

    A_al, B_al = align_skeletons(A_sel, B_sel)
    dists = np.linalg.norm(A_al - B_al, axis=1)  # per-joint distances

    avg = float(np.mean(dists))
    # Dynamic normalization: use median inter-joint baseline to scale into 0..100
    # Lower avg distance => higher score
    # Clamp to avoid negatives
    norm = float(np.median(dists) + 1e-6)
    score = max(0.0, min(100.0, 100.0 * (1.0 - (avg / (3.0 * norm)))))  # tune factor 3 for sensitivity
    return SimilarityResult(score=score, per_joint_dist=dists)

def grade_student(score: float) -> str:
    if score >= 80:
        return "Excellent"
    elif score >= 55:
        return "Good"
    elif score >= 35:
        return "Fair"
    else:
        return "Needs Improvement"

# Simple mapping of indices to names (MediaPipe Pose)
POSE_NAMES = [
    "nose","left_eye_inner","left_eye","left_eye_outer","right_eye_inner","right_eye","right_eye_outer",
    "left_ear","right_ear","mouth_left","mouth_right",
    "left_shoulder","right_shoulder","left_elbow","right_elbow","left_wrist","right_wrist",
    "left_pinky","right_pinky","left_index","right_index","left_thumb","right_thumb",
    "left_hip","right_hip","left_knee","right_knee","left_ankle","right_ankle",
    "left_heel","right_heel","left_foot_index","right_foot_index"
]

def suggest_improvement(A: np.ndarray, B: np.ndarray, per_joint_dist: np.ndarray) -> str:
    # per_joint_dist corresponds to STABLE_IDX order; map back to full names
    idx_local = int(np.argmax(per_joint_dist))
    idx_global = int(STABLE_IDX[idx_local])
    joint_name = POSE_NAMES[idx_global] if idx_global < len(POSE_NAMES) else f"joint_{idx_global}"
    return f"Focus on aligning your {joint_name}. Reduce deviation through targeted drills and mirror practice."

In [23]:
def save_debug_video(frames_bgr: List[np.ndarray], path: pathlib.Path, fps: int = 30) -> None:
    """Save frames to an MP4 for visual verification."""
    if not frames_bgr:
        return
    h, w = frames_bgr[0].shape[:2]
    path.parent.mkdir(parents=True, exist_ok=True)

    # 🔥 Overwrite if file exists
    if path.exists():
        path.unlink()

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(path), fourcc, fps, (w, h))
    for f in frames_bgr:
        writer.write(f)
    writer.release()

To get the frames per second - instead of manual frames

In [30]:
def get_video_frame_count(path: pathlib.Path) -> int:
    """Return total frame count for a video file."""
    cap = cv2.VideoCapture(str(path))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frames

In [34]:
def assess_taekwando(
    instructor_url: str,
    instructor_start_sec: int,
    instructor_end_sec: int,
    student_url: str,
    student_start_sec: int,
    student_end_sec: int,
    output_dir_path: str = r"C:\Users\arrun\OneDrive\Desktop\Computer Vision\Temp Taekwando project",
    max_frames: int | None = 600,   # cap frames for speed
    debug_viz: bool = True,
) -> None:
    outdir = pathlib.Path(output_dir_path)

    instructor_mp4 = download_clip(
        url=instructor_url,
        output_dir=outdir,
        start_sec=instructor_start_sec,
        end_sec=instructor_end_sec,
        basename="instructor",
    )
    if instructor_mp4 is None:
        raise RuntimeError("Failed to download instructor clip.")

    student_mp4 = download_clip(
        url=student_url,
        output_dir=outdir,
        start_sec=student_start_sec,
        end_sec=student_end_sec,
        basename="student",
    )
    if student_mp4 is None:
        raise RuntimeError("Failed to download student clip.")

    instr_frames = load_video_frames(instructor_mp4, max_frames=max_frames)
    stud_frames  = load_video_frames(student_mp4, max_frames=max_frames)

    instr_landmarks = extract_pose_landmarks(instr_frames)
    stud_landmarks  = extract_pose_landmarks(stud_frames)

    min_len = min(len(instr_landmarks), len(stud_landmarks))
    if min_len == 0:
        logging.info("No valid skeletons detected in one or both videos.")
        return

    instr_landmarks = instr_landmarks[:min_len]
    stud_landmarks  = stud_landmarks[:min_len]

    # Optional visualization: render skeleton overlays
    if debug_viz:
        instr_viz = []
        stud_viz  = []
        for i in range(min_len):
            instr_viz.append(draw_skeleton(instr_frames[i], instr_landmarks[i]))
            stud_viz.append(draw_skeleton(stud_frames[i],  stud_landmarks[i]))
        save_debug_video(instr_viz, outdir / "instructor_skeleton.mp4", fps=30)
        save_debug_video(stud_viz,  outdir / "student_skeleton.mp4",   fps=30)

    # Compute per-frame similarity and aggregate
    scores = []
    suggestions = []
    for A, B in zip(instr_landmarks, stud_landmarks):
        res = similarity_score(A, B)
        scores.append(res.score)
        suggestions.append(res.per_joint_dist)

    avg_score = float(np.mean(scores))
    grade = grade_student(avg_score)
    # Use the first frame’s largest deviation for a single suggestion (or aggregate)
    suggestion = suggest_improvement(instr_landmarks[0], stud_landmarks[0], suggestions[0])

    print("\n--- Taekwando Assessment ---")
    print(f"Frames compared: {min_len}")
    print(f"Average similarity score: {avg_score:.2f} / 100")
    print(f"Grade: {grade}")
    print(f"Suggested improvement: {suggestion}")
    print(f"Saved outputs to: {outdir}")

if __name__ == "__main__":
    # Instructor example (Segment: 1:20 to 2:00 -> 80 to 120 sec)
    instructor_url = "https://www.youtube.com/watch?v=Ot-rBhiIUKs"
    instructor_start = 80
    instructor_end   = 120

    # Student example ( specified segment: 2:53 to 3:28 -> 153 to 208 sec)
    student_url = "https://www.youtube.com/watch?v=etgxusKS0Do"
    student_start = 153
    student_end   = 208

    assess_taekwando(
        instructor_url=instructor_url,
        instructor_start_sec=instructor_start,
        instructor_end_sec=instructor_end,
        student_url=student_url,
        student_start_sec=student_start,
        student_end_sec=student_end,
        output_dir_path=r"C:\Users\arrun\OneDrive\Desktop\Computer Vision\Temp Taekwando project\Final Assesments",
        max_frames=None, # use all frames
        debug_viz=True,
    )

c:\Users\arrun\anaconda3\envs\tf-gpu\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


KeyboardInterrupt: 

In [35]:
def get_video_frame_count(path: pathlib.Path) -> int:
    """Return total frame count for a video file."""
    cap = cv2.VideoCapture(str(path))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frames

def assess_taekwando(
    instructor_url: str,
    instructor_start_sec: int,
    instructor_end_sec: int,
    student_url: str,
    student_start_sec: int,
    student_end_sec: int,
    output_dir_path: str = r"C:\Users\arrun\OneDrive\Desktop\Computer Vision\Temp Taekwando project",
    max_frames: int | None = None,   # auto-detect if None
    debug_viz: bool = True,
) -> None:
    outdir = pathlib.Path(output_dir_path)

    # Download clips
    instructor_mp4 = download_clip(
        url=instructor_url,
        output_dir=outdir,
        start_sec=instructor_start_sec,
        end_sec=instructor_end_sec,
        basename="instructor",
    )
    if instructor_mp4 is None:
        raise RuntimeError("Failed to download instructor clip.")

    student_mp4 = download_clip(
        url=student_url,
        output_dir=outdir,
        start_sec=student_start_sec,
        end_sec=student_end_sec,
        basename="student",
    )
    if student_mp4 is None:
        raise RuntimeError("Failed to download student clip.")

    # 🔥 Auto-detect frame counts if max_frames not provided
    if max_frames is None:
        instr_frames_total = get_video_frame_count(instructor_mp4)
        stud_frames_total  = get_video_frame_count(student_mp4)
        max_frames = min(instr_frames_total, stud_frames_total)
        print(f"Auto-set max_frames = {max_frames} (instr={instr_frames_total}, stud={stud_frames_total})")

    # Load frames
    instr_frames = load_video_frames(instructor_mp4, max_frames=max_frames)
    stud_frames  = load_video_frames(student_mp4, max_frames=max_frames)

    # Extract pose landmarks
    instr_landmarks = extract_pose_landmarks(instr_frames)
    stud_landmarks  = extract_pose_landmarks(stud_frames)

    min_len = min(len(instr_landmarks), len(stud_landmarks))
    if min_len == 0:
        logging.info("No valid skeletons detected in one or both videos.")
        return

    instr_landmarks = instr_landmarks[:min_len]
    stud_landmarks  = stud_landmarks[:min_len]

    # Optional visualization: render skeleton overlays
    if debug_viz:
        instr_viz = [draw_skeleton(f, l) for f, l in zip(instr_frames, instr_landmarks)]
        stud_viz  = [draw_skeleton(f, l) for f, l in zip(stud_frames,  stud_landmarks)]
        save_debug_video(instr_viz, outdir / "instructor_skeleton.mp4", fps=30)
        save_debug_video(stud_viz,  outdir / "student_skeleton.mp4",   fps=30)

    # Compute per-frame similarity and aggregate
    scores = []
    suggestions = []
    for A, B in zip(instr_landmarks, stud_landmarks):
        res = similarity_score(A, B)
        scores.append(res.score)
        suggestions.append(res.per_joint_dist)

    avg_score = float(np.mean(scores))
    grade = grade_student(avg_score)
    suggestion = suggest_improvement(instr_landmarks[0], stud_landmarks[0], suggestions[0])

    print("\n--- Taekwando Assessment ---")
    print(f"Frames compared: {min_len}")
    print(f"Average similarity score: {avg_score:.2f} / 100")
    print(f"Grade: {grade}")
    print(f"Suggested improvement: {suggestion}")
    print(f"Saved outputs to: {outdir}")

In [36]:
if __name__ == "__main__":
    # Instructor example (Segment: 1:20 to 2:00 -> 80 to 120 sec)
    instructor_url = "https://www.youtube.com/watch?v=Ot-rBhiIUKs"
    instructor_start = 80
    instructor_end   = 120

    # Student example ( specified segment: 2:53 to 3:28 -> 153 to 208 sec)
    student_url = "https://www.youtube.com/watch?v=etgxusKS0Do"
    student_start = 153
    student_end   = 208

    assess_taekwando(
        instructor_url=instructor_url,
        instructor_start_sec=instructor_start,
        instructor_end_sec=instructor_end,
        student_url=student_url,
        student_start_sec=student_start,
        student_end_sec=student_end,
        output_dir_path=r"C:\Users\arrun\OneDrive\Desktop\Computer Vision\Temp Taekwando project\Final Assesments",
        max_frames=None, # use all frames
        debug_viz=True,
    )

Auto-set max_frames = 3625 (instr=3625, stud=6510)


c:\Users\arrun\anaconda3\envs\tf-gpu\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



--- Taekwando Assessment ---
Frames compared: 3330
Average similarity score: 59.96 / 100
Grade: Good
Suggested improvement: Focus on aligning your right_wrist. Reduce deviation through targeted drills and mirror practice.
Saved outputs to: C:\Users\arrun\OneDrive\Desktop\Computer Vision\Temp Taekwando project\Final Assesments


Seems based on above max frames as 3625 , student videos are clipped from 6510, thats where the issue is

- Instructor and student clips are loaded to their own full lengths.
- Comparison is aligned only at the end (min_len), so you don’t lose extra frames unnecessarily.
- Skeleton overlays (instructor_skeleton.mp4, student_skeleton.mp4) now reflect the full duration of each clip, not truncated to the shorter one.


In [ ]:
def assess_taekwando(
    instructor_url: str,
    instructor_start_sec: int,
    instructor_end_sec: int,
    student_url: str,
    student_start_sec: int,
    student_end_sec: int,
    output_dir_path: str,
    max_frames: int | None = None,
    debug_viz: bool = True,
) -> None:
    outdir = pathlib.Path(output_dir_path)

    # Download clips
    instructor_mp4 = download_clip(instructor_url, outdir, instructor_start_sec, instructor_end_sec, "instructor")
    student_mp4    = download_clip(student_url, outdir, student_start_sec, student_end_sec, "student")

    if instructor_mp4 is None or student_mp4 is None:
        raise RuntimeError("Failed to download one of the clips.")

    # Auto-detect frame counts separately
    instr_total = get_video_frame_count(instructor_mp4)
    stud_total  = get_video_frame_count(student_mp4)

    # If max_frames is None, use full length for each
    instr_frames = load_video_frames(instructor_mp4, max_frames=instr_total if max_frames is None else max_frames)
    stud_frames  = load_video_frames(student_mp4,  max_frames=stud_total  if max_frames is None else max_frames)

    # Extract pose landmarks
    instr_landmarks = extract_pose_landmarks(instr_frames)
    stud_landmarks  = extract_pose_landmarks(stud_frames)

    # Align by shortest length for comparison
    min_len = min(len(instr_landmarks), len(stud_landmarks))
    instr_landmarks = instr_landmarks[:min_len]
    stud_landmarks  = stud_landmarks[:min_len]
    instr_frames    = instr_frames[:min_len]
    stud_frames     = stud_frames[:min_len]

    if min_len == 0:
        logging.info("No valid skeletons detected in one or both videos.")
        return

    # Visualization
    if debug_viz:
        instr_viz = [draw_skeleton(f, l) for f, l in zip(instr_frames, instr_landmarks)]
        stud_viz  = [draw_skeleton(f, l) for f, l in zip(stud_frames,  stud_landmarks)]
        save_debug_video(instr_viz, outdir / "instructor_skeleton.mp4", fps=30)
        save_debug_video(stud_viz,  outdir / "student_skeleton.mp4",   fps=30)

    # Similarity scoring
    scores = []
    suggestions = []
    for A, B in zip(instr_landmarks, stud_landmarks):
        res = similarity_score(A, B)
        scores.append(res.score)
        suggestions.append(res.per_joint_dist)

    avg_score = float(np.mean(scores))
    grade = grade_student(avg_score)
    suggestion = suggest_improvement(instr_landmarks[0], stud_landmarks[0], suggestions[0])

    print("\n--- Taekwando Assessment ---")
    print(f"Instructor frames: {len(instr_landmarks)} | Student frames: {len(stud_landmarks)}")
    print(f"Frames compared: {min_len}")
    print(f"Average similarity score: {avg_score:.2f} / 100")
    print(f"Grade: {grade}")
    print(f"Suggested improvement: {suggestion}")
    print(f"Saved outputs to: {outdir}")

Below test functions

In [26]:
!yt-dlp "https://www.youtube.com/watch?v=etgxusKS0Do" \
  -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4" \
  --download-sections "*176-208" \
  -o student_timestampcheck.mp4

[youtube] Extracting URL: https://www.youtube.com/watch?v=etgxusKS0Do
[youtube] etgxusKS0Do: Downloading webpage
[youtube] etgxusKS0Do: Downloading android sdkless player API JSON
[youtube] etgxusKS0Do: Downloading web safari player API JSON
[youtube] etgxusKS0Do: Downloading m3u8 information
[info] etgxusKS0Do: Downloading 1 format(s): 399+328
[info] etgxusKS0Do: Downloading 1 time ranges: 176.0-208.0
[download] Destination: student_timestampcheck.mp4

[download] 100% of    6.49MiB in 00:00:02 at 2.47MiB/s


Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'https://rr8---sn-bvvbaxivnuxq5uu-q4fz.googlevideo.com/videoplayback?expire=1764596285&ei=3EUtabDPOuHBy_sPlvn5gAQ&ip=2600%3A1700%3A78e0%3A81c0%3Ad1d3%3A592a%3A1ad9%3A567e&id=o-APCRM1CF9EtX-9-_uH0EhTdxnShuJllGsDpvNJ1q-g1_&itag=399&source=youtube&requiressl=yes&xpc=EgVo2aDSNQ%3D%3D&met=1764574684%2C&mh=or&mm=31%2C29&mn=sn-bvvbaxivnuxq5uu-q4fz%2Csn-q4flrnsd&ms=au%2Crdu&mv=m&mvi=8&pl=49&rms=au%2Cau&initcwndbps=2262500&bui=AdEuB5RfAe6dgUhUGkCpcHKPik4KIqTR0WBj6IW1B58gRYNSXnl2y_qbFo-obSU1Bic2W8WBVRW9mf_N&spc=6b0G_LxF1ZZv&vprv=1&svpuc=1&mime=video%2Fmp4&rqh=1&gir=yes&clen=20414188&dur=217.216&lmt=1708739343685301&mt=1764574133&fvip=3&keepalive=yes&fexp=51552689%2C51565115%2C51565682%2C51580968&c=ANDROID&txp=443G434&sparams=expire%2Cei%2Cip%2Cid%2Citag%2Csource%2Crequiressl%2Cxpc%2Cbui%2Cspc%2Cvprv%2Csvpuc%2Cmime%2Crqh%2Cgir%2Cclen%2Cdur%2Clmt&sig=AJfQdSswRAIgXwKfliyf-kWgIDq3g-3x6m3BxxurTHOj7ctlys_-dSsCIASBUq2caT7l1ITZEtlMJavZuArmrnA0TlCc-bVntG9S&lsparams